In [ ]:
!pip install python-dotenv langchain langchain-huggingface langchain-core langchain-community pinecone-client

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.6/50.6 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 408.0/408.0 kB 26.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 48.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 244.8/244.8 kB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.9/296.9 kB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.4/85.4 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 255.8/255.8 kB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.4/76.4 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.0/78.0 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.5/49.5 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.5/144.5 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/

In [ ]:
import os

os.environ['PINECONE_API_KEY'] = "Pine cone api"
os.environ['HUGGINGFACE_API_KEY'] = "Huggingface api key"

In [ ]:
from google.colab import files

uploaded = files.upload()


Saving routes.txt to routes.txt


In [ ]:
import os
from dotenv import load_dotenv
import pandas as pd
from math import radians, cos, sin, sqrt, atan2
from langchain.text_splitter import CharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFaceEndpoint
from langchain_core.prompts import PromptTemplate
from langchain.vectorstores import Pinecone as LangchainPinecone  # Import from langchain
from langchain.schema.runnable import RunnablePassthrough
from langchain.schema.output_parser import StrOutputParser
import pinecone
from pinecone import ServerlessSpec

class Document:
    def __init__(self, page_content: str, metadata: dict = None):
        self.page_content = page_content
        self.metadata = metadata if metadata is not None else {}

class SimpleTextLoader:
    def __init__(self, file_path: str):
        self.file_path = file_path
    def load(self):
        with open(self.file_path, 'r', encoding='utf-8') as file:
            text = file.read()
        return [Document(page_content=text, metadata={})]

class ChatBot:
    def __init__(self):
        load_dotenv()

        # Load route data from CSV
        try:
            self.routes_df = pd.read_csv('https://dagshub.com/Omdena/VITBhopalUniversity_ChatbotforBRTSNavigation/raw/99c2e8d2883dd9faaa68ed60d5405dd40e77c456/src/tasks/task-2/Routes/all_routes_combined.csv')
        except Exception as e:
            print(f"Error loading routes CSV: {e}")
            return

        # Initialize embeddings and vector store
        embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")

        # Initialize Pinecone
        pinecone_api_key = os.getenv('PINECONE_API_KEY')
        pinecone.Pinecone(api_key=pinecone_api_key, environment='us-west1-gcp')

        # Create an instance of Pinecone class
        pc = pinecone.Pinecone()

        index_name = "langchain-demo"
        # if index_name not in pc.list_indexes():  # Now accessed through the instance
        #     pc.create_index(
        #         name=index_name,
        #         dimension=768,
        #         metric="cosine",
        #         spec=ServerlessSpec(
        #             cloud='gcp',
        #             region='lowa (us-central1)'
        #         )  # Specify cloud and region as needed
        #     )

        text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=4)
        self.loader = SimpleTextLoader('./routes.txt')
        self.documents = self.loader.load()
        docs = text_splitter.split_documents(self.documents)

        # Use Langchain's Pinecone vector store
        self.vectorstore = LangchainPinecone.from_documents(docs, embeddings, index_name=index_name)

        # Use a suitable model for route assistance
        huggingface_api_key = os.getenv('HUGGINGFACE_API_KEY')
        if not huggingface_api_key:
            print("Hugging Face API key missing. Check your .env file.")
            return
        repo_id = "mistralai/Mixtral-8x7B-Instruct-v0.1"
        self.llm = HuggingFaceEndpoint(
            repo_id=repo_id,
            temperature=0.7,
            top_k=30,
            max_new_tokens=128,
            huggingfacehub_api_token=huggingface_api_key,
            model_kwargs={}
        )

        # Route assistant prompt with context for guidance
        template = """
        You are a route assistant for the BRTS system. Users will ask you questions about routes, stops, and how to get to their destinations.
        Use the following route information to assist them. If you don't know the answer, just say you don't know.
        Context: {context}
        Question: {question}
        Answer:
        """
        self.prompt = PromptTemplate(
            template=template,
            input_variables=["context", "question"]
        )
        self.rag_chain = (
            {"context": self.vectorstore.as_retriever(), "question": RunnablePassthrough()}
            | self.prompt
            | self.llm
            | StrOutputParser()
        )

    def haversine(self, lat1, lon1, lat2, lon2):
        # Haversine formula to calculate distance between two points on Earth
        lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
        dlon = lon2 - lon1
        dlat = lat2 - lat1
        a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
        c = 2 * atan2(sqrt(a), sqrt(1-a))
        r = 6371  # Radius of Earth in kilometers
        return c * r

    def find_closest_bus_station(self, current_lat, current_lon):
        # Calculate distance for each bus station and find the closest
        self.routes_df['distance'] = self.routes_df.apply(
            lambda row: self.haversine(current_lat, current_lon, row['Latitude'], row['Longitude']),
            axis=1
        )
        closest_station = self.routes_df.loc[self.routes_df['distance'].idxmin()]
        return closest_station

    def ask(self, user_input):
        try:
            if "navigate" in user_input.lower():
                parts = user_input.split(" ")
                if len(parts) < 2 or "," not in parts[1]:
                    return "Please provide coordinates in the format: 'navigate latitude,longitude'"

                lat_lon = parts[1].split(",")
                current_lat, current_lon = float(lat_lon[0]), float(lat_lon[1])
                closest_station = self.find_closest_bus_station(current_lat, current_lon)
                return (
                    f"Here is the nearest bus stop based on your location.\n\n"
                    f"**Closest Stop:** {closest_station['Station']}\n"
                    f"**Location:** {closest_station['Latitude']}, {closest_station['Longitude']}\n"
                    f"[Navigate using Google Maps](https://www.google.com/maps/dir/{current_lat},{current_lon}/{closest_station['Latitude']},{closest_station['Longitude']})"
                )
            else:
                result = self.rag_chain.invoke(user_input)
                return result
        except ValueError as e:
            return f"Invalid input: {e}. Please ensure coordinates are in 'latitude,longitude' format."
        except Exception as e:
            print(f"Error during invocation: {e}")
            return "Sorry, I couldn't process your request."

if __name__ == "__main__":
    bot = ChatBot()
    user_input = input("Ask me anything: ")
    result = bot.ask(user_input)
    print(result)


/usr/local/lib/python3.10/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `huggingface-cli` if you want to set the git credential as well.
Token is valid (permission: fineGrained).
Your token has been saved to /root/.cache/huggingface/token
Login successful
Ask me anything: tell me something about brts

        The Bhopal Bus Rapid Transit System (BRTS), also known as MyBus, is a public transportation system in Bhopal, Madhya Pradesh, India. It was funded by the Central Government under its flagship JnNURM program and primarily catered to the needs of the Central Business Districts (CBDs). The system launched in 2006 with a fleet of 30 buses and expanded to 225 buses, including both AC and non-AC low-floor models. However, due to traffic issues, the Government of Madh


In [ ]:
import os
from dotenv import load_dotenv
import pandas as pd
from math import radians, cos, sin, sqrt, atan2
from langchain.text_splitter import CharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFaceEndpoint
from langchain_core.prompts import PromptTemplate
from langchain.vectorstores import Pinecone as LangchainPinecone  # Import from langchain
from langchain.schema.runnable import RunnablePassthrough
from langchain.schema.output_parser import StrOutputParser
import pinecone
from pinecone import ServerlessSpec

class Document:
    def __init__(self, page_content: str, metadata: dict = None):
        self.page_content = page_content
        self.metadata = metadata if metadata is not None else {}

class SimpleTextLoader:
    def __init__(self, file_path: str):
        self.file_path = file_path
    def load(self):
        with open(self.file_path, 'r', encoding='utf-8') as file:
            text = file.read()
        return [Document(page_content=text, metadata={})]

class ChatBot:
    def __init__(self):
        load_dotenv()

        # Load route data from CSV
        try:
            self.routes_df = pd.read_csv('https://dagshub.com/Omdena/VITBhopalUniversity_ChatbotforBRTSNavigation/raw/99c2e8d2883dd9faaa68ed60d5405dd40e77c456/src/tasks/task-2/Routes/all_routes_combined.csv')
        except Exception as e:
            print(f"Error loading routes CSV: {e}")
            return

        # Initialize embeddings and vector store
        embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")

        # Initialize Pinecone
        pinecone_api_key = os.getenv('PINECONE_API_KEY')
        pinecone.Pinecone(api_key=pinecone_api_key, environment='us-west1-gcp')

        # Create an instance of Pinecone class
        pc = pinecone.Pinecone()

        index_name = "langchain-demo"
        # if index_name not in pc.list_indexes():  # Now accessed through the instance
        #     pc.create_index(
        #         name=index_name,
        #         dimension=768,
        #         metric="cosine",
        #         spec=ServerlessSpec(
        #             cloud='gcp',
        #             region='lowa (us-central1)'
        #         )  # Specify cloud and region as needed
        #     )

        text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=4)
        self.loader = SimpleTextLoader('./routes.txt')
        self.documents = self.loader.load()
        docs = text_splitter.split_documents(self.documents)

        # Use Langchain's Pinecone vector store
        self.vectorstore = LangchainPinecone.from_documents(docs, embeddings, index_name=index_name)

        # Use a suitable model for route assistance
        huggingface_api_key = os.getenv('HUGGINGFACE_API_KEY')
        if not huggingface_api_key:
            print("Hugging Face API key missing. Check your .env file.")
            return
        repo_id = "mistralai/Mixtral-8x7B-Instruct-v0.1"
        self.llm = HuggingFaceEndpoint(
            repo_id=repo_id,
            temperature=0.7,
            top_k=30,
            max_new_tokens=128,
            huggingfacehub_api_token=huggingface_api_key,
            model_kwargs={}
        )

        # Route assistant prompt with context for guidance
        template = """
        You are a route assistant for the BRTS system. Users will ask you questions about routes, stops, and how to get to their destinations.
        Use the following route information to assist them. If you don't know the answer, just say you don't know.
        Context: {context}
        Question: {question}
        Answer:
        """
        self.prompt = PromptTemplate(
            template=template,
            input_variables=["context", "question"]
        )
        self.rag_chain = (
            {"context": self.vectorstore.as_retriever(), "question": RunnablePassthrough()}
            | self.prompt
            | self.llm
            | StrOutputParser()
        )

    def haversine(self, lat1, lon1, lat2, lon2):
        # Haversine formula to calculate distance between two points on Earth
        lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
        dlon = lon2 - lon1
        dlat = lat2 - lat1
        a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
        c = 2 * atan2(sqrt(a), sqrt(1-a))
        r = 6371  # Radius of Earth in kilometers
        return c * r

    def find_closest_bus_station(self, current_lat, current_lon):
        # Calculate distance for each bus station and find the closest
        self.routes_df['distance'] = self.routes_df.apply(
            lambda row: self.haversine(current_lat, current_lon, row['Latitude'], row['Longitude']),
            axis=1
        )
        closest_station = self.routes_df.loc[self.routes_df['distance'].idxmin()]
        return closest_station

    def ask(self, user_input):
        try:
            if "navigate" in user_input.lower():
                parts = user_input.split(" ")
                if len(parts) < 2 or "," not in parts[1]:
                    return "Please provide coordinates in the format: 'navigate latitude,longitude'"

                lat_lon = parts[1].split(",")
                current_lat, current_lon = float(lat_lon[0]), float(lat_lon[1])
                closest_station = self.find_closest_bus_station(current_lat, current_lon)
                return (
                    f"Here is the nearest bus stop based on your location.\n\n"
                    f"**Closest Stop:** {closest_station['Station']}\n"
                    f"**Location:** {closest_station['Latitude']}, {closest_station['Longitude']}\n"
                    f"[Navigate using Google Maps](https://www.google.com/maps/dir/{current_lat},{current_lon}/{closest_station['Latitude']},{closest_station['Longitude']})"
                )
            else:
                result = self.rag_chain.invoke(user_input)
                return result
        except ValueError as e:
            return f"Invalid input: {e}. Please ensure coordinates are in 'latitude,longitude' format."
        except Exception as e:
            print(f"Error during invocation: {e}")
            return "Sorry, I couldn't process your request."

if __name__ == "__main__":
    bot = ChatBot()
    user_input = input("Ask me anything: ")
    result = bot.ask(user_input)
    print(result)


/usr/local/lib/python3.10/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `huggingface-cli` if you want to set the git credential as well.
Token is valid (permission: fineGrained).
Your token has been saved to /root/.cache/huggingface/token
Login successful
Ask me anything: navigate 23.2732,77.3634
Here is the nearest bus stop based on your location.

**Closest Stop:** Halalpur Bus Stand
**Location:** 23.27326584, 77.36399078
[Navigate using Google Maps](https://www.google.com/maps/dir/23.2732,77.3634/23.27326584,77.36399078)
